# **Data Set details + Reproducibility**
You will use the AG News dataset (4-class news topic classification). It comes with an official train/test split.

Training set: 120,000 examples
Test set: 7,600 examples

The random seed will be set at "2026".

In [ ]:
from datasets import load_dataset
dataset = load_dataset("ag_news")
# print(dataset)

In [ ]:
# Create a validation split
train_and_validation = dataset['train'].train_test_split(test_size=0.1, seed=2026)

train_set = train_and_validation['train']
val_set = train_and_validation['test']
test_set = dataset['test']

In [ ]:
import numpy as np
import torch
import random

RANDOM_SEED = 2026
np.random.default_rng(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
  worker_seed = RANDOM_SEED + worker_id
  np.random.seed(worker_seed)
  random.seed(worker_seed)

# **Tokenization for LSTM**
* Go through the training set's texts and split them into words (or word-like tokens).
*   Assign each with a unique integer ID.
*   Have PAD for padding and UNK for unknown words.




In [ ]:
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
import torch

# 1. Build vocabular
all_text = [t for t in train_set['text']]
words = [w for text in all_text for w in text.lower().split()]
word_freq = Counter(words)

vocab = {w: i + 2 for i, (w, count) in enumerate(word_freq.most_common()) if count >= 2}
vocab['PAD'] = 0
vocab['UNK'] = 1

# 2️. Tokenizer
def tokenize_lstm(text):
    return [vocab.get(w, vocab['UNK']) for w in text.lower().split()][:128]

# 3️. Training data
tokenized_train = [torch.tensor(tokenize_lstm(t)) for t in train_set['text']]
train_labels = torch.tensor(train_set['label'])
train_lengths = torch.tensor([len(seq) for seq in tokenized_train])
padded_train = pad_sequence(tokenized_train, batch_first=True, padding_value=vocab['PAD'])

# 4️. Validation data
tokenized_val = [torch.tensor(tokenize_lstm(t)) for t in val_set['text']]
val_labels = torch.tensor(val_set['label'])
val_lengths = torch.tensor([len(seq) for seq in tokenized_val])
padded_val = pad_sequence(tokenized_val, batch_first=True, padding_value=vocab['PAD'])

# 5️. Test data
tokenized_test = [torch.tensor(tokenize_lstm(t)) for t in test_set['text']]
test_labels = torch.tensor(test_set['label'])
test_lengths = torch.tensor([len(seq) for seq in tokenized_test])
padded_test = pad_sequence(tokenized_test, batch_first=True, padding_value=vocab['PAD'])

# **LSTM trained From Scratch**
*   Embedding(vocab_size, d_emb)
*   LSTM(d_emb, d_hidden, num_layers=..., dropout=..., bidirectional=(True/False))
*   Mean pooling with mask
*   Linear(d_hidden, 4) to produce logits over 4 classes

In [ ]:
# Model constants
EMB_DIM = 128
HIDDEN_DIM = 256
NUM_LAYER = 2 # (or 2 if stable)
DROPOUT = 0.3
BIDIRECTIONAL = True

# Training loop constants
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
NUM_EPOCHS = 6


In [ ]:
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class LSTM(nn.Module):
  def __init__(self, vocab_size, d_emb, d_hidden, num_layers, dropout, bidirectional):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, d_emb)

    self.lstm = nn.LSTM(
        input_size = d_emb,
        hidden_size = d_hidden,
        num_layers = num_layers,
        dropout = dropout,
        bidirectional = bidirectional,
        batch_first = True
    )

    # Account for output size if bidirectional
    direction_factor = 2 if bidirectional else 1

    self.fc = nn.Linear(d_hidden * direction_factor, 4)


  def forward(self, x, lengths):
    # x: (batch, seq_len), lengths: original lengths before padding
    x = self.embedding(x)                                             # (batch, seq_len, d_emb)
    packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)

    packed_output, (h_n, c_n) = self.lstm(packed_x)                   # output: (batch, seq_len, hidden*dir)

    # Unpack
    output, _ = pad_packed_sequence(packed_output, batch_first=True)  # (batch, seq_len, hidden*dir)

    # Masked mean pooling
    batch_size, seq_len, _ = output.shape
    mask = torch.arange(seq_len, device=lengths.device)[None, :] < lengths[:, None] # Positions less than the length are real tokens (not masked), so they are True.

    masked_output = output * mask.unsqueeze(-1)                       # (batch, seq_len, hidden*dir)

    """
    word1 → [1, 0]
    word2 → [0, 2]
    word3 → [3, 1]

    1. sum over words (token dimension)
    [1, 0] + [0, 2] + [3, 1] = [4, 3]

    2. divide by number of words (3)
    [4/3, 3/3] = [1.33, 1.0]

    """
    x = masked_output.sum(dim=1) / lengths.unsqueeze(1)

    # Classifier
    x = self.fc(x)  # (batch, 4)
    return x


# **Training Loop**
* Using suggested hyperparameters
* Modifications (if any): Tested with 2 layers and changed hidden dimensions to 128


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils as nn_utils

# Create dataset + loader
train_dataset = TensorDataset(padded_train, train_lengths, train_labels)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TensorDataset(padded_val, val_lengths, val_labels)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = LSTM(len(vocab), EMB_DIM, HIDDEN_DIM, NUM_LAYER, DROPOUT, BIDIRECTIONAL)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

for epoch in range(NUM_EPOCHS):

  model.train()
  total_loss = 0

  for batch_idx, (x_batch, lengths_batch, y_batch) in enumerate(train_loader):

    """ Verify that padding and tokenization works correctly
    if epoch == 0 and batch_idx == 0:
        print("x_batch shape:", x_batch.shape)
        print("x_batch[0]:", x_batch[0])
        print("length:", lengths_batch[0])
        print("label:", y_batch[0])
    """

    # Forward
    x_batch = x_batch.to(device)
    lengths_batch = lengths_batch.to(device)
    y_batch = y_batch.to(device)

    outputs = model(x_batch, lengths_batch)
    loss = criterion(outputs, y_batch)

    # Backward
    optimizer.zero_grad()
    loss.backward()

    # Gradient clipping
    nn_utils.clip_grad_norm_(model.parameters(), 1.0)

    # Update
    optimizer.step()

    total_loss += loss.item()

  # Validation
  model.eval()

  total_val_loss = 0
  total_correct = 0
  total_samples = 0

  with torch.no_grad():
    for x_batch, lengths_batch, y_batch in val_loader:
      x_batch = x_batch.to(device)
      lengths_batch = lengths_batch.to(device)
      y_batch = y_batch.to(device)

      output = model(x_batch, lengths_batch)
      loss = criterion(output, y_batch)
      total_val_loss += loss.item()

      preds = output.argmax(dim=1)
      total_correct += (preds == y_batch).sum().item()
      total_samples += y_batch.size(0)

  avg_val_loss = total_val_loss / len(val_loader)
  val_accuracy = total_correct / total_samples

  avg_loss = total_loss / len(train_loader)
  print(f"Epoch {epoch+1}: Train Loss = {avg_loss:.4f}, Val Loss = {avg_val_loss:.4f}, Val Accuracy = {val_accuracy:.2f}")

# **Testing the model**


In [ ]:
# Make sure model is on the correct device
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

test_correct = 0

test_dataset = TensorDataset(padded_test, test_lengths, test_labels)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

store_misclassified_labels = []

with torch.no_grad():
    for batch_idx, (x_batch, lengths_batch, y_batch) in enumerate(test_loader):
        # Move batch to same device as model
        x_batch = x_batch.to(device)
        lengths_batch = lengths_batch.to(device)
        y_batch = y_batch.to(device)

        output = model(x_batch, lengths_batch)
        preds = output.argmax(dim=1)

        # Count correct predictions
        test_correct += (preds == y_batch).sum().item()

        # Find misclassified indices
        wrong_idxs = (preds != y_batch).nonzero(as_tuple=True)[0]

        # Store misclassified examples
        for i in wrong_idxs:
            sample_idx = batch_idx * BATCH_SIZE + i.item()
            store_misclassified_labels.append((
                test_set['text'][sample_idx],
                y_batch[i].item(),
                preds[i].item()
            ))

# Compute accuracy
test_accuracy = test_correct / len(test_dataset)
print(f'Test accuracy: {test_accuracy:.4f}')

# Display first 10 misclassified examples
for text, true_label, pred_label in store_misclassified_labels[:10]:
    print("Text:", text)
    print("True label:", true_label)
    print("Predicted label:", pred_label)
    print("-"*50)